# GEDI Canopy Height Pipeline and The Bayesian Statistical Prediction Scenario

This notebook processes NASA GEDI Level 2B HDF5 files stored in S3 through four sequential phases:

| Phase | Description |
|---|---|
| **Phase 1** | High-performance batch extraction of GEDI shots from S3 |
| **Phase 1a** | Year parsing from GEDI filenames |
| **Phase 1b** | Harmonization to the SMAP 9 km grid |
| **Phase 1c** | Spatial join to assign shots to study-area jurisdictions |

**Source:** `s3://central-virginia-tree-canopy-project/GEDI/GEDI02_B/002/`  
**Final output:** `s3://central-virginia-tree-canopy-project/gedi02B-county-summary/virginia_gedi02B_county_summary.csv`

## Cell 1 — Install Dependencies

In [1]:
import subprocess, sys

# Pin the entire boto stack atomically — all four packages must be compatible
boto_stack = [
    "botocore==1.43.0",
    "boto3==1.43.0",
    "s3fs==2026.6.0",
]

# Install boto stack with --no-deps to prevent pip from overriding the pins
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                       "--force-reinstall"] + boto_stack)

# Install all other dependencies normally
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                       "h5py", "numpy", "pandas", "pymc", "arviz", "ipywidgets", "seaborn"])

print("Boto Stack and all dependencies installed.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
awscli 1.45.42 requires botocore==1.43.42, but you have botocore 1.43.0 which is incompatible.
awscli 1.45.42 requires s3transfer<0.20.0,>=0.19.0, but you have s3transfer 0.17.1 which is incompatible.
sagemaker 2.257.3 requires attrs<26,>=24, but you have attrs 26.1.0 which is incompatible.


Boto Stack and all dependencies installed.


## Cell 2 — Imports

In [2]:
import os
import numpy as np
import pandas as pd
import pymc as pm
import arviz as az

print("Imports successful.")

Imports successful.


## Cell 3 — Configuration

When necessary, edit this cell to change S3 paths, bounding box, grid resolution, or target jurisdictions.

In [3]:
# ── S3 settings ───────────────────────────────────────────────────────────────
S3_BUCKET                = "central-virginia-tree-canopy-project/"
GEDI02A_COUNTY_S3_PREFIX = "gedi-county-summary/"
SMAP_S3_PREFIX           = "dashboard-data/"

# ── Local output directory ────────────────────────────────────────────────────
OUTPUT_DIR        = "/home/ec2-user/SageMaker/gedi_output"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Configuration loaded.")
print(f"  GEDI02_A source file  : s3://{S3_BUCKET}/{GEDI02A_COUNTY_S3_PREFIX}")
print(f"  Output dir   : {OUTPUT_DIR}")
print(f"  S3 output    : s3://{S3_BUCKET}/{GEDI02A_COUNTY_S3_PREFIX}")

Configuration loaded.
  GEDI02_A source file  : s3://central-virginia-tree-canopy-project//gedi-county-summary/
  Output dir   : /home/ec2-user/SageMaker/gedi_output
  S3 output    : s3://central-virginia-tree-canopy-project//gedi-county-summary/


## Cell 4 — Load a JSON Data File

In [4]:
# Define the S3 path
s3_json_uri = "s3://" + S3_BUCKET + SMAP_S3_PREFIX + "merged_smap_gedi.json"

print(f"s3_json_uri: {s3_json_uri}")

# Load directly into a pandas DataFrame
df_matrix = pd.read_json(s3_json_uri)

# Preview the matrix data
df_matrix.head()

s3_json_uri: s3://central-virginia-tree-canopy-project/dashboard-data/merged_smap_gedi.json


,year,jurisdiction,canopy_height_mean_m,sm_mean_m3m3,sm_min,sm_max,sm_std,sm_mean_lag1,sm_mean_lag2
0,2019,Albemarle,19.675812,0.379557,0.314088,0.436462,0.027698,NaN,NaN
1,2020,Albemarle,18.465603,0.307442,0.229515,0.389360,0.034866,0.379557,NaN
2,2021,Albemarle,18.793875,0.287135,0.214029,0.363267,0.033156,0.307442,0.379557
3,2022,Albemarle,18.639362,0.288770,0.218431,0.366198,0.033531,0.287135,0.307442
4,2023,Albemarle,13.609988,0.271359,0.197718,0.351615,0.036388,0.288770,0.287135


## Cell 5 — Define future soil moisture scenarios (2024 to 2028) to represent assumptions about regional weather trends

In [5]:
scenarios = {
    "Severe Drought": {
        "sm_mean": [0.26, 0.24, 0.23, 0.22, 0.21],  # Steady decline through 2028
        "sm_std":  [0.038, 0.040, 0.042, 0.045, 0.045] # Higher volatility
    },
    "Climate Recovery": {
        "sm_mean": [0.29, 0.32, 0.34, 0.35, 0.36],  # Steady wet recovery
        "sm_std":  [0.033, 0.031, 0.029, 0.028, 0.028] # Lower, stable volatility
    }
}

future_years = [2024, 2025, 2026, 2027, 2028]
jurisdictions = list(jurisdiction_names)  # Inherited from previous training setup
n_chains_draws = 4000  # Total trace samples from your model (e.g., 4 chains x 1000 draws)

# Extract posterior parameter samples from your trained MCMC trace
posterior_alpha = trace.posterior["alpha"].values.reshape(n_chains_draws, len(jurisdictions))
posterior_beta_mean = trace.posterior["beta_sm_mean"].values.flatten()
posterior_beta_std = trace.posterior["beta_sm_std"].values.flatten()
posterior_gamma = trace.posterior["gamma_autoreg"].values.flatten()
posterior_sigma = trace.posterior["sigma_residual"].values.flatten()

# 2. Get the baseline initialization state (Year 2023 actual values)
# For Charlottesville, which lacks 2023 data, we default back to its 2022 state
init_canopy = {}
for j in jurisdictions:
    j_data = df[df["jurisdiction"] == j]
    if 2023 in j_data["year"].values:
        init_canopy[j] = j_data[j_data["year"] == 2023]["canopy_height_mean_m"].values[0]
    else:
        init_canopy[j] = j_data[j_data["year"] == 2022]["canopy_height_mean_m"].values[0]

# Dictionary to hold the final forecast outputs
simulation_results = {}




NameError: name 'jurisdiction_names' is not defined